# Education extension: fertility and female secondary enrollment

This is a separate 2001–2023 analysis. Gross female secondary enrollment is a proxy for education, not adult attainment, and may exceed 100%. Missing enrollment data changes the sample, so this notebook compares models on identical rows and country-held-out folds. All code for this extension is visible below.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'data').is_dir():
    raise FileNotFoundError('Open this notebook from the project folder or notebooks folder.')
panel = pd.read_csv(ROOT/'data/processed/country_year_panel.csv')
print('Loaded', len(panel), 'economy-year rows')

Loaded 14322 economy-year rows


## Coverage and complete cases

Keep the education period separate from the main 1990–2024 cohort. GDP must be positive for the log transform. No values are filled or interpolated.

In [2]:
period = panel.loc[panel.year.between(2001, 2023)].copy()
period['log_gdp'] = np.log(period.gdp_pc.where(period.gdp_pc.gt(0)))
base = period.dropna(subset=['fertility', 'log_gdp', 'mortality'])
education = period.dropna(subset=['fertility', 'log_gdp', 'mortality', 'education']).copy()
coverage = pd.DataFrame({
    'year': sorted(period.year.unique()),
    'fertility': period.groupby('year').fertility.count().values,
    'GDP/person': period.groupby('year').gdp_pc.count().values,
    'under-five mortality': period.groupby('year').mortality.count().values,
    'female secondary enrollment': period.groupby('year').education.count().values,
})
display(pd.DataFrame({'sample': ['Core complete', 'Core + education complete'],
                      'country-years': [len(base), len(education)],
                      'economies': [base.iso3.nunique(), education.iso3.nunique()]}))
display(coverage.tail())
fig = px.line(coverage, x='year', y=['fertility', 'GDP/person', 'under-five mortality', 'female secondary enrollment'],
              title='WDI data coverage during the education period',
              labels={'value': 'Economies with an observed value', 'variable': 'Indicator'})
fig.show()

,sample,country-years,economies
0,Core complete,4404,194
1,Core + education complete,3137,189


,year,fertility,GDP/person,under-five mortality,female secondary enrollment
18,2019,217,208,196,141
19,2020,217,208,196,144
20,2021,217,208,196,148
21,2022,217,208,196,140
22,2023,217,203,196,144


## Observed association

The scatter plot uses only economies with all four values in 2023. Color denotes current WDI region metadata. The relationship is descriptive.

In [3]:
snapshot = education.loc[education.year.eq(2023)].copy()
snapshot['region'] = snapshot.region.str.strip()
display(pd.DataFrame({'feature': ['log GDP/person', 'under-five mortality', 'female secondary enrollment'],
                      'Spearman correlation with fertility': [snapshot.fertility.corr(snapshot[x], method='spearman')
                          for x in ['log_gdp', 'mortality', 'education']]}).round(3))
fig = px.scatter(snapshot, x='education', y='fertility', color='region', hover_name='country',
                 title='Female secondary enrollment and fertility, 2023',
                 labels={'education': 'Gross female secondary enrollment (%)', 'fertility': 'Births per woman'})
fig.show()

,feature,Spearman correlation with fertility
0,log GDP/person,-0.752
1,under-five mortality,0.799
2,female secondary enrollment,-0.573


## Paired prediction comparison

Both feature sets use the same 3,137 rows and identical five country-held-out folds. Ridge scales variables within each training fold. The forest uses fixed settings. Improvement means lower held-out MAE; it does not identify an educational effect.

In [4]:
feature_sets = {'GDP + mortality': ['log_gdp', 'mortality'],
                'GDP + mortality + enrollment': ['log_gdp', 'mortality', 'education']}
model_factories = {'Ridge': lambda: make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
                   'Random forest': lambda: RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
                                                                  random_state=42, n_jobs=-1)}
results = []
folds = GroupKFold(n_splits=5)
for fold, (train_idx, test_idx) in enumerate(folds.split(education, education.fertility, education.iso3), start=1):
    train, test = education.iloc[train_idx], education.iloc[test_idx]
    for model_name, factory in model_factories.items():
        for feature_name, columns in feature_sets.items():
            model = factory()
            model.fit(train[columns], train.fertility)
            predicted = model.predict(test[columns])
            results.append({'fold': fold, 'model': model_name, 'features': feature_name,
                            'MAE': mean_absolute_error(test.fertility, predicted),
                            'RMSE': np.sqrt(mean_squared_error(test.fertility, predicted)),
                            'n_test_rows': len(test)})
results = pd.DataFrame(results)
display(results.groupby(['model', 'features'])[['MAE', 'RMSE']].mean().round(3))
gains = results.pivot(index=['fold', 'model'], columns='features', values='MAE').reset_index()
gains['MAE improvement'] = gains['GDP + mortality'] - gains['GDP + mortality + enrollment']
display(gains[['fold', 'model', 'MAE improvement']].round(3))
fig = px.bar(gains, x='fold', y='MAE improvement', color='model', barmode='group',
             title='Change in held-out prediction error after adding enrollment',
             labels={'MAE improvement': 'MAE reduction (births per woman)'})
fig.show()

MAE   RMSE
model         features                                  
Random forest GDP + mortality               0.538  0.741
              GDP + mortality + enrollment  0.488  0.678
Ridge         GDP + mortality               0.529  0.705
              GDP + mortality + enrollment  0.519  0.681

features,fold,model,MAE improvement
0,1,Random forest,0.095
1,1,Ridge,-0.001
2,2,Random forest,0.104
3,2,Ridge,0.020
4,3,Random forest,0.051
5,3,Ridge,-0.003
6,4,Random forest,-0.015
7,4,Ridge,0.003
8,5,Random forest,0.015
9,5,Ridge,0.032


## Interpretation and limits

Enrollment may add predictive information on this later, incomplete sample. It is a gross enrollment rate, so values over 100% can be valid. Results are sensitive to missing data and cohort choice. Compare the paired feature sets only within this notebook; their scores are not directly comparable with the main 1990–2024 models. These country-level associations do not establish causation.